# RVC Colab Nvidia

Set the runtime to a Nvidia GPU before running these cells. Google Drive is the default storage target.

All training and inference is driven from this notebook via the `rvc_colab` Python module — no web UI.

In [ ]:
!nvidia-smi

In [ ]:
# Mount Drive first so models, datasets, logs, and outputs persist.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Fresh Colab setup.
!rm -rf /content/RVC-Colab-Nvidia
!git clone https://github.com/Axel-Jalonen/RVC-Colab-Nvidia.git /content/RVC-Colab-Nvidia
%cd /content/RVC-Colab-Nvidia
!python -m pip install -U pip setuptools wheel
!pip install -r requirements-colab.txt

In [ ]:
# Downloads HuBERT, RMVPE, and pretrained training weights into Drive by default. Add --no-pretrained for inference only.
!python tools/download_colab_assets.py

## Load the API

Importing `rvc_colab` configures the Drive paths (`/content/drive/MyDrive/RVC-Colab/...`) and probes the GPU. Run `rvc.paths()` to see where models, datasets, and outputs are read from.

In [ ]:
import sys
sys.path.insert(0, '/content/RVC-Colab-Nvidia')

import rvc_colab as rvc

rvc.runtime_status()
rvc.paths()

## Inference

Drop `.pth` voice models in `/content/drive/MyDrive/RVC-Colab/models/` and (optional) `.index` files in `/content/drive/MyDrive/RVC-Colab/indices/`.

In [ ]:
rvc.list_models(), rvc.list_indices()

In [ ]:
model = rvc.load_model('my_voice.pth', index='my_voice.index')

sr, audio = rvc.convert(
    model,
    '/content/drive/MyDrive/RVC-Colab/datasets/input.wav',
    transpose=0,
    f0_method='rmvpe',
    index_rate=0.75,
    output_path='/content/drive/MyDrive/RVC-Colab/outputs/converted.wav',
)

from IPython.display import Audio
Audio(audio, rate=sr)

In [ ]:
# Batch convert a folder.
rvc.convert_batch(
    model,
    input_dir='/content/drive/MyDrive/RVC-Colab/datasets/inbox',
    output_dir='/content/drive/MyDrive/RVC-Colab/outputs/batch',
    output_format='wav',
)

## Training

Put your training audio (any number of `.wav`/`.mp3`/`.flac` files) in a folder under `/content/drive/MyDrive/RVC-Colab/datasets/<exp_name>/`. `train_voice` runs preprocess → feature extraction → train → build index. Subprocess output streams into the cell.

In [ ]:
rvc.train_voice(
    exp_name='my_voice',
    dataset_dir='/content/drive/MyDrive/RVC-Colab/datasets/my_voice',
    sr='40k',
    version='v2',
    use_f0=True,
    f0_method='rmvpe_gpu',
    total_epoch=100,
    save_epoch=10,
    batch_size=8,
)

Need finer control? Call the steps individually:

```python
rvc.preprocess_dataset('my_voice', '.../datasets/my_voice', sr='40k')
rvc.extract_features('my_voice', f0_method='rmvpe_gpu', version='v2')
rvc.train_model('my_voice', sr='40k', total_epoch=100)
rvc.build_index('my_voice', version='v2')
```